In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
datadir = "/content/drive/My Drive/"

Mounted at /content/drive


In [ ]:
df = pd.read_json(datadir + 'execution-dataset.json')

In [ ]:
df.rename(columns={'code': 'normalized_code'}, inplace=True)

In [ ]:
import re

def clean_code(code: str) -> str:
    """Remove comments, imports, docstrings, and normalize whitespace."""
    if not isinstance(code, str):
        return ""
    code = re.sub(r"(#.*?$|//.*?$|/\*.*?\*/|\"\"\".*?\"\"\"|from.*?$|import.*?$)", "", code, flags=re.MULTILINE | re.DOTALL)
    code = re.sub(r"\s+", " ", code).strip()
    return code

df["normalized_code"] = df["normalized_code"].apply(clean_code)
df["actual_code"] = df["actual_code"].apply(clean_code)

In [ ]:
df = df[(df["normalized_code"].notnull() & df["normalized_code"].str.strip().ne("") & df["ip_variables"].astype(str).ne("{}") & df["op_variables"].astype(str).ne("{}"))].reset_index(drop=True)

In [ ]:
df

,normalized_code,actual_code,ip_variables,op_variables,imports,execution_time,disk_usage,memory_usage
0,pandas.core.frame.DataFrame = pandas.read_csv(...,data_train = pd.read_csv('/kaggle/input/titani...,"{'data_train': None, 'data_test': None}",{'data_train': {'id': 'pandas.core.frame.DataF...,{'pd': 'pandas'},0.012623,"{'peak_disk': 1514211.52734375, 'increment': 0.0}","{'peak_memory': 284.70909, 'memory_increment':..."
1,pandas.core.frame.DataFrame.info(),data_train.info(),{'data_train': {'id': 'pandas.core.frame.DataF...,{'data_train': {'id': 'pandas.core.frame.DataF...,{},0.006703,"{'peak_disk': 1514211.52734375, 'increment': 0.0}","{'peak_memory': 285.92593, 'memory_increment':..."
2,pandas.core.frame.DataFrame.head(10),data_train.head(10),{'data_train': {'id': 'pandas.core.frame.DataF...,{'data_train': {'id': 'pandas.core.frame.DataF...,{},0.000279,"{'peak_disk': 1514211.52734375, 'increment': 0.0}","{'peak_memory': 285.93641999999994, 'memory_in..."
3,def prepocess(df): df['Age'] = df['Age'].filln...,def prepocess(df): df['Age'] = df['Age'].filln...,"{'df': None, 'emb_map': None, 'prepocess': Non...","{'df': None, 'emb_map': None, 'data_train': {'...",{'np': 'numpy'},0.004129,"{'peak_disk': 1514211.52734375, 'increment': 0.0}","{'peak_memory': 286.07278999999994, 'memory_in..."
4,def func_alone(df): df['Alone'] = numpy.where(...,"def func_alone(df): df[""Alone""] = np.where(df[...","{'df': None, 'func_alone': None, 'data_train':...","{'df': None, 'data_train': {'id': 'pandas.core...",{'np': 'numpy'},0.001857,"{'peak_disk': 1514211.52734375, 'increment': 0.0}","{'peak_memory': 286.08328, 'memory_increment':..."
...,...,...,...,...,...,...,...,...
14487,torch.device = torch.device('cuda' if torch.cu...,"device = torch.device(""cuda"" if torch.cuda.is_...",{'device': None},"{'device': {'id': 'torch.device', 'shape': Non...",{'torch': 'torch'},8.547286,"{'peak_disk': 1534223.0703125, 'increment': 0....","{'peak_memory': 1333.18459, 'memory_increment'..."
14488,"dict = {0: 'NEGATIVE', 1: 'POSITIVE'} dict = {...","id2label={0: ""NEGATIVE"", 1: ""POSITIVE""} label2...","{'id2label': None, 'label2id': None, 'model_na...","{'id2label': {'id': 'dict', 'shape': 2, 'size'...",{'GPT2ForSequenceClassification': 'transformer...,2.892857,"{'peak_disk': 1534223.06640625, 'increment': 0.0}","{'peak_memory': 1824.95579, 'memory_increment'..."
14489,pandas.core.frame.DataFrame = pandas.read_csv(...,train_df_csv = pd.read_csv('/kaggle/input/nlp-...,"{'train_df_csv': None, 'test_df': None, 'sampl...",{'train_df_csv': {'id': 'pandas.core.frame.Dat...,"{'pd': 'pandas', 'train_test_split': 'sklearn....",0.040583,"{'peak_disk': 1534223.06640625, 'increment': 0.0}","{'peak_memory': 1495.6642, 'memory_increment':..."
14490,datasets.arrow_dataset.Dataset = datasets.Data...,train_dataset = Dataset. eval_dataset = Datase...,"{'train_dataset': None, 'train_df': {'id': 'pa...",{'train_dataset': {'id': 'datasets.arrow_datas...,"{'Dataset': 'datasets.Dataset', 'DatasetDict':...",8.430541,"{'peak_disk': 1534223.06640625, 'increment': 0.0}","{'peak_memory': 1646.80412, 'memory_increment'..."


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import RobertaTokenizer, RobertaModel

cluster_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
cluster_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")

cluster_encoder = cluster_encoder.to(device)
cluster_encoder.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [ ]:
def embed_token(code):
  inputs = cluster_tokenizer(code, return_tensors="pt", truncation=True, padding=True, max_length=256)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = cluster_encoder(**inputs)
  return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

In [ ]:
df = pd.read_json('processed-execution-dataset.json')

In [ ]:
from sklearn.preprocessing import normalize

embeddings = np.array([embed_token(code) for code in df['normalized_code'].tolist()])
embeddings = normalize(embeddings)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, metric='euclidean')
df['cluster'] = clusterer.fit_predict(embeddings)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cluster_to_label = {}
for cluster_id in df['cluster'].unique():
    idxs = df[df['cluster'] == cluster_id].index.tolist()
    cluster_embeds = embeddings[idxs]
    centroid = cluster_embeds.mean(axis=0)
    sims = cosine_similarity([centroid], cluster_embeds)[0]
    rep_idx = idxs[sims.argmax()]
    cluster_to_label[cluster_id] = df.loc[rep_idx, 'normalized_code']

df['class_name'] = df['cluster'].map(cluster_to_label)

In [ ]:
df.to_json('processed-execution-dataset.json', orient='records', indent=2)